# 🔧 Notebook 03 — Declarative Pipelines Deep Dive

**What you'll learn:**
- Replace 4+ LLM round-trips with a single pipeline call
- MongoDB-style filtering with logical operators (`$and`, `$or`, `$not`)
- Aggregations: group_by, metrics (count, mean, sum, median, stddev)
- `save_as`: store pipeline results as new browsable workspaces
- Nested field access with dot notation

**Why pipelines matter:**
Each tool call = one LLM turn = latency + tokens + cost. A 5-step exploration
(search → filter → sort → limit → select) costs ~$0.10 in API calls and 10+ seconds.
A pipeline does it in ONE call: ~$0.02, 2 seconds.

In [ ]:
import json
from ctxtual import Ctx, MemoryStore
from ctxtual.utils import paginator, text_search, filter_set, pipeline

ctx = Ctx(store=MemoryStore())
pager = paginator(ctx, "orders")
search = text_search(ctx, "orders")
filters = filter_set(ctx, "orders")
pipe = pipeline(ctx, "orders")

# Simulated e-commerce dataset
import random
random.seed(42)

categories = ["Electronics", "Clothing", "Books", "Home", "Sports", "Food"]
regions = ["US-East", "US-West", "EU-West", "EU-East", "APAC", "LATAM"]
statuses = ["delivered", "shipped", "processing", "cancelled", "returned"]

@ctx.producer(workspace_type="orders", toolsets=[pager, search, filters, pipe])
def load_orders() -> list[dict]:
    """Load order data from the warehouse."""
    orders = []
    for i in range(10_000):
        orders.append({
            "order_id": f"ORD-{i:06d}",
            "customer": f"Customer_{i % 500}",
            "category": random.choice(categories),
            "region": random.choice(regions),
            "amount": round(random.uniform(5, 500), 2),
            "quantity": random.randint(1, 20),
            "status": random.choice(statuses),
            "priority": random.choice(["low", "medium", "high"]),
            "year": random.choice([2022, 2023, 2024]),
            "tags": random.sample(["fragile", "express", "gift", "bulk", "subscription"], k=random.randint(0, 3)),
        })
    return orders

ref = load_orders()
wid = ref["workspace_id"]
print(f"Loaded {ref['item_count']:,} orders")
print(f"Schema: {json.dumps(ref.get('item_schema', {}), indent=2)}")

## 1. Basic Pipeline: Filter → Sort → Limit

In [ ]:
# Top 10 highest-value Electronics orders from 2024
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"category": "Electronics", "year": 2024}},
        {"sort": {"field": "amount", "order": "desc"}},
        {"limit": 10},
        {"select": ["order_id", "customer", "amount", "status"]},
    ],
})
print(f"Top 10 Electronics orders in 2024 (from {ref['item_count']:,} total):\n")
for item in result["items"]:
    print(f"  {item['order_id']}  ${item['amount']:>7.2f}  {item['status']:<12}  {item['customer']}")
print(f"\nResult count: {result['count']}")

## 2. MongoDB-Style Operators

In [ ]:
# Complex filter: high-priority orders over $200 that are NOT delivered
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {
            "priority": "high",
            "amount": {"$gt": 200},
            "status": {"$nin": ["delivered", "cancelled"]},
        }},
        {"sort": {"field": "amount", "order": "desc"}},
        {"limit": 5},
    ],
})
print(f"High-priority, >$200, not delivered/cancelled: {result['count']} found\n")
for item in result["items"]:
    print(f"  {item['order_id']}  ${item['amount']:.2f}  status={item['status']}  region={item['region']}")

In [ ]:
# Logical operators: ($or, $and, $not)
# Orders from APAC OR LATAM, in 2024, with "express" in tags
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {
            "$or": [
                {"region": "APAC"},
                {"region": "LATAM"},
            ],
            "year": 2024,
            "tags": {"$contains": "express"},
        }},
        {"count": True},
    ],
})
print(f"APAC/LATAM express orders in 2024: {result['count']}")

## 3. Aggregations: group_by + Metrics

In [ ]:
# Revenue by category
result = ctx.dispatch_tool_call("orders_aggregate", {
    "workspace_id": wid,
    "group_by": "category",
    "metrics": {
        "total_revenue": "sum:amount",
        "avg_order": "mean:amount",
        "order_count": "count",
        "max_order": "max:amount",
    },
})
print("=== Revenue by Category ===\n")
print(f"{'Category':<15} {'Orders':>8} {'Total Rev':>12} {'Avg Order':>10} {'Max Order':>10}")
print("-" * 60)
# Sort by total revenue
groups = sorted(result["groups"], key=lambda g: g["total_revenue"], reverse=True)
for g in groups:
    print(f"{g['category']:<15} {g['order_count']:>8} ${g['total_revenue']:>11,.2f} ${g['avg_order']:>9.2f} ${g['max_order']:>9.2f}")
print(f"\nTotal groups: {result['group_count']}")

In [ ]:
# Pipeline-based aggregation: group_by inside a pipeline step
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"year": 2024}},
        {"group_by": {
            "field": "region",
            "metrics": {
                "n": "count",
                "revenue": "sum:amount",
                "avg": "mean:amount",
            },
        }},
        {"sort": {"field": "revenue", "order": "desc"}},
    ],
})
print("=== 2024 Revenue by Region ===\n")
for item in result["items"]:
    print(f"  {item['region']:<10} {item['n']:>5} orders  ${item['revenue']:>10,.2f} total  ${item['avg']:>.2f} avg")

## 4. `flatten` — Explode List Fields

In [ ]:
# Tag frequency analysis: flatten tags → group → sort
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"flatten": "tags"},
        {"group_by": {
            "field": "tags",
            "metrics": {"count": "count", "avg_amount": "mean:amount"},
        }},
        {"sort": {"field": "count", "order": "desc"}},
    ],
})
print("=== Tag Frequency Analysis ===\n")
print(f"{'Tag':<15} {'Count':>8} {'Avg Amount':>12}")
print("-" * 40)
for item in result["items"]:
    print(f"{item['tags']:<15} {item['count']:>8} ${item['avg_amount']:>11.2f}")

## 5. `save_as` — Create Derived Workspaces

In [ ]:
# Save high-value orders as a new workspace for further exploration
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"amount": {"$gte": 400}, "status": "delivered"}},
        {"sort": {"field": "amount", "order": "desc"}},
    ],
    "save_as": "premium_orders",
})
print("=== Saved as new workspace ===")
print(f"Status: {result['status']}")
print(f"Items:  {result['item_count']}")
print(f"Schema: {json.dumps(result.get('item_schema', {}), indent=2)}")

In [ ]:
# Now paginate the derived workspace!
premium_wid = result["premium_orders"]
page = ctx.dispatch_tool_call("orders_paginate", {
    "workspace_id": premium_wid,
    "page": 0,
    "size": 5,
})
print(f"Premium orders workspace: {page['result']['total']} items\n")
for item in page["result"]["items"]:
    print(f"  {item['order_id']}  ${item['amount']:.2f}  {item['region']}")

## 6. Advanced: Multi-Sort, Sample, Unique, Text Search

In [ ]:
# Deduplicate by customer, keeping highest-value order
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"sort": {"field": "amount", "order": "desc"}},
        {"unique": "customer"},
        {"limit": 10},
        {"select": ["customer", "order_id", "amount"]},
    ],
})
print("=== Top spending customers (1 order each) ===")
for item in result["items"]:
    print(f"  {item['customer']:<15}  {item['order_id']}  ${item['amount']:.2f}")

In [ ]:
# Random sample with reproducible seed
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"filter": {"category": "Books"}},
        {"sample": {"n": 5, "seed": 42}},
        {"select": ["order_id", "customer", "amount"]},
    ],
})
print("=== Random sample of 5 Book orders (seed=42) ===")
for item in result["items"]:
    print(f"  {item['order_id']}  {item['customer']}  ${item['amount']:.2f}")

In [ ]:
# Text search within pipeline
result = ctx.dispatch_tool_call("orders_pipe", {
    "workspace_id": wid,
    "steps": [
        {"search": {"query": "Customer_42", "fields": ["customer"]}},
        {"sort": {"field": "amount", "order": "desc"}},
        {"select": ["order_id", "customer", "amount", "category"]},
    ],
})
print(f"=== Orders for Customer_42: {result['count']} found ===")
for item in result["items"][:5]:
    print(f"  {item['order_id']}  ${item['amount']:.2f}  {item['category']}")

## Pipeline Operator Reference

| Operator | Example | Description |
|----------|---------|-------------|
| `filter` | `{"filter": {"year": {"$gte": 2023}}}` | MongoDB-style: `$gt`, `$gte`, `$lt`, `$lte`, `$ne`, `$in`, `$nin`, `$contains`, `$startswith`, `$regex`, `$exists`, `$or`, `$and`, `$not` |
| `search` | `{"search": {"query": "ML", "fields": ["title"]}}` | Case-insensitive text search |
| `sort` | `{"sort": {"field": "score", "order": "desc"}}` | Single or multi-field sort |
| `select` | `{"select": ["title", "author"]}` | Keep only listed fields |
| `exclude` | `{"exclude": ["raw_blob"]}` | Remove listed fields |
| `limit` | `{"limit": 10}` | First N items |
| `skip` | `{"skip": 5}` | Skip first N |
| `slice` | `{"slice": [5, 15]}` | Items `[5:15]` |
| `sample` | `{"sample": {"n": 10, "seed": 42}}` | Random sample |
| `unique` | `{"unique": "author"}` | Deduplicate by field |
| `flatten` | `{"flatten": "tags"}` | Expand list fields |
| `group_by` | `{"group_by": {"field": "cat", "metrics": {"n": "count"}}}` | Aggregation |
| `count` | `{"count": true}` | Terminal: return `{"count": N}` |

**Next:** [04_openai_agent_loop.ipynb](04_openai_agent_loop.ipynb) — full agent loop with OpenAI SDK